# DC-Ops: Export YOLOv8n-seg to ExecuTorch with QNN HTP Backend

Compiles the model for Snapdragon 8 Elite (SM8750) Hexagon HTP (NPU).

QNN compilation requires **Linux x86_64** — that's why we use Colab.

**Runtime → T4 GPU** (need Linux, GPU optional but speeds up quantization)

In [ ]:
# 1. Install ExecuTorch from source with QNN support
!git clone --depth 1 https://github.com/pytorch/executorch.git
%cd executorch
!git submodule sync
!git submodule update --init --recursive --depth 1

In [ ]:
# 2. Install ExecuTorch + QNN SDK
!pip install -q py-cpuinfo ultralytics
!bash install_requirements.sh
!pip install -e . --no-build-isolation

# Download QNN SDK for Linux
!python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null || true

import os
# Find QNN SDK path
qnn_path = !python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null
QNN_SDK_ROOT = qnn_path[-1].strip() if qnn_path else ''
os.environ['QNN_SDK_ROOT'] = QNN_SDK_ROOT
print(f'QNN SDK: {QNN_SDK_ROOT}')

In [ ]:
# 3. Build QNN backend Python bindings
import subprocess, os

# Build PyQnnManagerAdaptor
!python setup.py build_ext --inplace 2>&1 | tail -5

# Alternative: use cmake to build QNN backend
!cmake . -DEXECUTORCH_BUILD_QNN=ON -DQNN_SDK_ROOT=$QNN_SDK_ROOT -Bcmake-out-qnn 2>&1 | tail -10
!cmake --build cmake-out-qnn -j$(nproc) 2>&1 | tail -10

In [ ]:
# 4. Download the trained model from Hugging Face
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id='abhijitbetigeri/dc-ops-dataset',
    filename='models/dc_ops_yolov8n_seg_v3.pt',
    repo_type='dataset',
)
print(f'Model downloaded: {model_path}')

In [ ]:
# 5. Export with QNN HTP backend for SM8750
import torch
from ultralytics import YOLO

from executorch.backends.qualcomm.export_utils import (
    build_executorch_binary,
    make_quantizer,
    QnnConfig,
)
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype
from executorch.backends.qualcomm.serialization.qc_schema import (
    QnnExecuTorchBackendType,
    QcomChipset,
)

# Load YOLOv8n-seg
yolo = YOLO(model_path)
torch_model = yolo.model.eval()

# Create calibration data (random inputs for quantization)
calib_inputs = [(torch.randn(1, 3, 640, 640),) for _ in range(10)]

# Configure QNN for Snapdragon 8 Elite HTP
qnn_config = QnnConfig(
    soc_model=QcomChipset.SM8750,
    backend=QnnExecuTorchBackendType.kHtpBackend,
)

# Quantizer for HTP (INT8 or INT16)
quantizer = make_quantizer(
    quant_dtype=QuantDtype.use_8a8w,
    backend=qnn_config.backend,
    soc_model=qnn_config.soc_model,
)

# Build the .pte with QNN HTP backend
print('Building ExecuTorch binary with QNN HTP backend...')
build_executorch_binary(
    model=torch_model,
    qnn_config=qnn_config,
    file_name='dc_ops_yolov8n_seg_qnn',
    dataset=calib_inputs,
    custom_quantizer=quantizer,
)

import os
pte_path = 'dc_ops_yolov8n_seg_qnn.pte'
print(f'Exported: {pte_path} ({os.path.getsize(pte_path) / 1e6:.1f} MB)')
print('Backend: QNN HTP (Hexagon Tensor Processor)')
print('Target: SM8750 (Snapdragon 8 Elite)')

In [ ]:
# 6. Download the QNN .pte
from google.colab import files
files.download('dc_ops_yolov8n_seg_qnn.pte')